<a href="https://colab.research.google.com/github/Shukra865/NorthStar_Data_Project/blob/main/NorthStar_R_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# --- ENVIRONMENT SETUP ---
# Installing the required SQL library for data manipulation
if (!require("sqldf")) install.packages("sqldf", repos = "http://cran.us.r-project.org")
library(sqldf)

# --- DATA LOADING ---
# Importing the primary 'orders' dataset for analysis
# Note: Ensure orders.csv is visible in the sidebar before running
orders <- read.csv("orders.csv")

# --- INITIAL DATA AUDIT ---
# Using SQL to identify naming inconsistencies in the pickup_zone column
# This check identifies duplicate entries caused by case-sensitivity
zone_audit <- sqldf("
  SELECT pickup_zone, COUNT(*) as record_count
  FROM orders
  GROUP BY pickup_zone
  ORDER BY record_count DESC
")

# Display the audit table for the project report
print(zone_audit)

   pickup_zone record_count
1         East          104
2        South          103
3         EAST          103
4    RiverSide           86
5      Airport           85
6         WEST           84
7          Ctr           80
8      Central           79
9      CENTRAL           79
10       SOUTH           78
11        West           71
12   Riverside           65
13       north           64
14       NORTH           60
15     AIRPORT           59
16       North           50


In [9]:
# --- DATA CLEANING ---
# Standardizing the 'pickup_zone' column to Proper Case
# This ensures 'NORTH', 'north', and 'North' are treated as a single entity
orders$pickup_zone <- ifelse(tolower(orders$pickup_zone) == "central", "Central",
                      ifelse(tolower(orders$pickup_zone) == "south", "South",
                      ifelse(tolower(orders$pickup_zone) == "north", "North",
                      ifelse(tolower(orders$pickup_zone) == "east", "East",
                      ifelse(tolower(orders$pickup_zone) == "west", "West",
                      ifelse(tolower(orders$pickup_zone) == "airport", "Airport",
                      ifelse(tolower(orders$pickup_zone) == "riverside", "Riverside",
                      ifelse(tolower(orders$pickup_zone) == "ctr", "Central", orders$pickup_zone))))))))

# --- VERIFICATION ---
# Re-running the audit to confirm the data is now clean
cleaned_audit <- sqldf("
  SELECT pickup_zone, COUNT(*) as total_orders
  FROM orders
  GROUP BY pickup_zone
  ORDER BY total_orders DESC
")

# Display the standardized results for the assignment
print("Standardized Pickup Zones:")
print(cleaned_audit)

[1] "Standardized Pickup Zones:"
  pickup_zone total_orders
1     Central          238
2        East          207
3       South          181
4       North          174
5        West          155
6   Riverside          151
7     Airport          144


In [14]:
# --- DATA PREPARATION ---
deliveries <- read.csv("deliveries.csv")
master_data <- merge(orders, deliveries, by = "order_id")

# --- DATE-TIME PARSING ---
# We use 'tryFormats' to tell R to look for common date styles
# This solves the "unambiguous format" error
master_data$order_time <- as.POSIXct(master_data$order_created_at,
                                     format = "%Y-%m-%d %H:%M:%S",
                                     tryFormats = c("%Y-%m-%d %H:%M:%S", "%d/%m/%Y %H:%M", "%m/%d/%Y %H:%M"))

master_data$delivery_time <- as.POSIXct(master_data$delivery_completed_at,
                                        format = "%Y-%m-%d %H:%M:%S",
                                        tryFormats = c("%Y-%m-%d %H:%M:%S", "%d/%m/%Y %H:%M", "%m/%d/%Y %H:%M"))

# --- EFFICIENCY CALCULATION ---
# Calculating duration in minutes
master_data$duration_mins <- as.numeric(difftime(master_data$delivery_time,
                                                master_data$order_time,
                                                units = "mins"))

# Important: Removing rows where times couldn't be calculated
master_data <- master_data[!is.na(master_data$duration_mins), ]

# --- PERFORMANCE REPORT ---
performance_report <- sqldf("
  SELECT pickup_zone,
         ROUND(AVG(duration_mins), 2) as avg_delivery_mins,
         COUNT(*) as total_deliveries
  FROM master_data
  GROUP BY pickup_zone
  ORDER BY avg_delivery_mins DESC
")

print("Zone Performance Report (Standardized Time):")
print(performance_report)

[1] "Zone Performance Report (Standardized Time):"
  pickup_zone avg_delivery_mins total_deliveries
1     Central            728.33              168
2   Riverside            693.18              117
3        East            672.57              155
4       North            642.45              133
5       South            641.10              135
6        West            640.52              111
7     Airport            628.20              112


In [15]:
# --- CRITICAL DELAY ANALYSIS ---
# Identifying orders that exceeded the 12-hour (720 min) threshold
critical_delays <- sqldf("
  SELECT pickup_zone,
         COUNT(*) as delay_count
  FROM master_data
  WHERE duration_mins > 720
  GROUP BY pickup_zone
  ORDER BY delay_count DESC
")

print("Zones with Deliveries Exceeding 12 Hours:")
print(critical_delays)

[1] "Zones with Deliveries Exceeding 12 Hours:"
  pickup_zone delay_count
1     Central          74
2        East          60
3       South          47
4   Riverside          46
5     Airport          44
6       North          42
7        West          40
